In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEMPEST MASTER CELL v2 — Complete pipeline with all reviewer fixes
# Changes from v1:
#   • Statistical test: 5 seeds → 15 seeds
#   • Paired t-test → Wilcoxon signed-rank test (scipy.stats.wilcoxon)
#   • Statistical boxplot updated to show 15 points
#   • All other steps unchanged from working v1
# ═══════════════════════════════════════════════════════════════════════════════

import os, json, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, SimpleRNN, Dense,
                                      Dropout, BatchNormalization,
                                      Conv1D, MaxPooling1D, Flatten)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from scipy import stats
from collections import Counter

# ── JSON helper ───────────────────────────────────────────────────────────────
def save_json(obj, path):
    def convert(o):
        if isinstance(o, dict):            return {k: convert(v) for k,v in o.items()}
        if isinstance(o, list):            return [convert(v) for v in o]
        if isinstance(o, (np.integer,)):   return int(o)
        if isinstance(o, (np.floating,)):  return float(o)
        if isinstance(o, np.ndarray):      return o.tolist()
        return o
    with open(path, 'w') as f:
        json.dump(convert(obj), f, indent=2)

for d in ['/kaggle/working/data/raw', '/kaggle/working/data/raw2',
          '/kaggle/working/data/splits', '/kaggle/working/results',
          '/kaggle/working/figures']:
    os.makedirs(d, exist_ok=True)

print("="*70)
print("TEMPEST MASTER CELL v2 — Full pipeline")
print("="*70)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: DOWNLOAD DATASETS
# ══════════════════════════════════════════════════════════════════════════════
print("\n[1/13] Downloading datasets...")
os.system('kaggle datasets download -d ang3loliveira/malware-analysis-datasets-api-call-sequences --unzip -p /kaggle/working/data/raw/ -q')
os.system('kaggle datasets download -d gautamkarat/api-call-sequences --unzip -p /kaggle/working/data/raw2/ -q')
print("✅ Done.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: BUILD VOCABULARY — map integer IDs to real API names
# ══════════════════════════════════════════════════════════════════════════════
print("\n[2/13] Building API vocabulary...")
vocab_map = {}
try:
    with open('/kaggle/working/data/raw/sample_analysis_data.txt', 'r') as f:
        raw_text = f.read()
    api_names_in_text = raw_text.lower().strip().split()
    unique_sorted = sorted(set(api_names_in_text))
    for idx, name in enumerate(unique_sorted):
        vocab_map[idx] = name
    print(f"  Vocabulary built: {len(vocab_map)} API names")
    print(f"  Sample: { {k: vocab_map[k] for k in list(vocab_map)[:5]} }")
except Exception as e:
    print(f"  Could not build vocabulary: {e}")
    vocab_map = {i: f"api_{i}" for i in range(307)}

save_json({str(k): v for k,v in vocab_map.items()},
          '/kaggle/working/results/vocab_map.json')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: LOAD + PREPROCESS
# ══════════════════════════════════════════════════════════════════════════════
print("\n[3/13] Loading and preprocessing...")

df1 = pd.read_csv('/kaggle/working/data/raw/dynamic_api_call_sequence_per_malware_100_0_306.csv')
df2 = pd.read_csv('/kaggle/working/data/raw2/new_dataset.csv')

X1 = df1[[f't_{i}' for i in range(100)]].values.astype(np.int32)
y1 = df1['malware'].values

seq_cols2 = [c for c in df2.columns if c not in ['hash', 'malware']]
X2_raw = df2[seq_cols2].values[:, :100].astype(np.float32)
col_med = np.nanmedian(X2_raw, axis=0)
nan_mask = np.isnan(X2_raw)
X2_raw[nan_mask] = np.take(col_med, np.where(nan_mask)[1])
X2_raw = np.nan_to_num(X2_raw, nan=0.0).astype(np.int32)
X2 = X2_raw + int(X1.max()) + 1
y2 = df2['malware'].values

X = np.vstack([X1, X2])
y = np.concatenate([y1, y2])
num_api = int(X.max()) + 1

print(f"  Combined: {X.shape} | Malware: {y.sum()} | Benign: {(y==0).sum()}")
print(f"  Vocab size: {num_api}")

# Split FIRST — no leakage
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train_raw, X_val, y_train_raw, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1, random_state=42, stratify=y_temp)

print(f"  Test — Malware: {y_test.sum()} | Benign: {(y_test==0).sum()}")

# SMOTE on training only — after split to prevent leakage
print("  Applying SMOTE on training data only...")
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)
print(f"  Train after SMOTE: {X_train.shape}")

# Balanced held-out test set (undersample malware to match benign count)
ben_idx_test = np.where(y_test == 0)[0]
mal_idx_test = np.where(y_test == 1)[0]
n_ben = len(ben_idx_test)
np.random.seed(42)
mal_idx_sub = np.random.choice(mal_idx_test, size=n_ben, replace=False)
bal_idx     = np.concatenate([ben_idx_test, mal_idx_sub])
np.random.RandomState(42).shuffle(bal_idx)
X_test_bal  = X_test[bal_idx]
y_test_bal  = y_test[bal_idx]
print(f"  Balanced test — {n_ben} malware + {n_ben} benign = {len(y_test_bal)} total")
print(f"  'Flag everything' baseline on balanced test: {y_test_bal.sum()/len(y_test_bal):.1%}")

np.save('/kaggle/working/data/splits/X_train.npy', X_train)
np.save('/kaggle/working/data/splits/X_val.npy',   X_val)
np.save('/kaggle/working/data/splits/X_test.npy',  X_test)
np.save('/kaggle/working/data/splits/y_train.npy', y_train)
np.save('/kaggle/working/data/splits/y_val.npy',   y_val)
np.save('/kaggle/working/data/splits/y_test.npy',  y_test)
print("✅ Splits saved.")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL BUILDERS
# ══════════════════════════════════════════════════════════════════════════════
def build_lstm(vocab_size, seq_len=100, emb_dim=32):
    m = Sequential([
        Embedding(input_dim=vocab_size+1, output_dim=emb_dim, input_length=seq_len),
        LSTM(128, return_sequences=True),
        BatchNormalization(), Dropout(0.3),
        LSTM(64, return_sequences=False),
        BatchNormalization(), Dropout(0.3),
        Dense(32, activation='relu'), Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                       tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall')])
    return m

def build_ff(vocab_size, seq_len=100, emb_dim=32):
    m = Sequential([
        Embedding(input_dim=vocab_size+1, output_dim=emb_dim, input_length=seq_len),
        Flatten(),
        Dense(256, activation='relu'), Dropout(0.3),
        Dense(128, activation='relu'), Dropout(0.3),
        Dense(64,  activation='relu'),
        Dense(1,   activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m

def get_callbacks(path='/kaggle/working/best_model.keras'):
    return [
        EarlyStopping(monitor='val_auc', patience=5,
                      restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=0),
        ModelCheckpoint(path, monitor='val_auc', save_best_only=True,
                        mode='max', verbose=0)
    ]

def evaluate_both(model_or_probs, X_full, y_full, X_bal, y_bal, bal_idx,
                  is_model=True):
    if is_model:
        prob_full = model_or_probs.predict(X_full, verbose=0).flatten()
    else:
        prob_full = np.array(model_or_probs)
    prob_bal  = prob_full[bal_idx]
    pred_full = (prob_full > 0.5).astype(int)
    pred_bal  = (prob_bal  > 0.5).astype(int)
    auc_full  = roc_auc_score(y_full, prob_full)
    auc_bal   = roc_auc_score(y_bal,  prob_bal)
    rep_full  = classification_report(y_full, pred_full,
                                       target_names=['Benign','Malware'],
                                       output_dict=True)
    rep_bal   = classification_report(y_bal, pred_bal,
                                       target_names=['Benign','Malware'],
                                       output_dict=True)
    return prob_full, auc_full, rep_full, auc_bal, rep_bal

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: TRAIN MAIN TEMPEST MODEL
# ══════════════════════════════════════════════════════════════════════════════
print("\n[4/13] Training main TEMPEST model...")
tf.random.set_seed(42); np.random.seed(42)
model = build_lstm(num_api)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, batch_size=256,
    callbacks=get_callbacks(), verbose=1
)

prob_main, auc_main, rep_main, auc_main_bal, rep_main_bal = evaluate_both(
    model, X_test, y_test, X_test_bal, y_test_bal, bal_idx)

print(f"\n✅ TEMPEST — Full test AUC: {auc_main:.4f} | Acc: {rep_main['accuracy']:.4f}")
print(f"   TEMPEST — Balanced test AUC: {auc_main_bal:.4f} | Acc: {rep_main_bal['accuracy']:.4f}")
print(f"   Benign F1 (balanced): {rep_main_bal['Benign']['f1-score']:.4f}")
print(classification_report(y_test, (prob_main>0.5).astype(int),
                             target_names=['Benign','Malware']))

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: BASELINES
# ══════════════════════════════════════════════════════════════════════════════
print("\n[5/13] Training baselines...")
results = {}
results['TEMPEST (Emb+LSTM)'] = {
    'prob': prob_main.tolist(),
    'auc_full': auc_main, 'report_full': rep_main,
    'auc_bal':  auc_main_bal, 'report_bal': rep_main_bal
}

# Random Forest
print("  Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
_, auc_rf, rep_rf, auc_rf_bal, rep_rf_bal = evaluate_both(
    rf_prob, X_test, y_test, X_test_bal, y_test_bal, bal_idx, is_model=False)
results['Random Forest'] = {'prob': rf_prob.tolist(),
    'auc_full': auc_rf, 'report_full': rep_rf,
    'auc_bal': auc_rf_bal, 'report_bal': rep_rf_bal}
print(f"    RF  — Full AUC: {auc_rf:.4f} | Balanced AUC: {auc_rf_bal:.4f}")

# SVM
print("  Training SVM...")
idx_svm = np.random.choice(len(X_train), 10000, replace=False)
svm = SVC(probability=True, kernel='rbf', random_state=42, class_weight='balanced')
svm.fit(X_train[idx_svm], y_train[idx_svm])
svm_prob = svm.predict_proba(X_test)[:, 1]
_, auc_svm, rep_svm, auc_svm_bal, rep_svm_bal = evaluate_both(
    svm_prob, X_test, y_test, X_test_bal, y_test_bal, bal_idx, is_model=False)
results['SVM (RBF)'] = {'prob': svm_prob.tolist(),
    'auc_full': auc_svm, 'report_full': rep_svm,
    'auc_bal': auc_svm_bal, 'report_bal': rep_svm_bal}
print(f"    SVM — Full AUC: {auc_svm:.4f} | Balanced AUC: {auc_svm_bal:.4f}")

# CNN
print("  Training CNN...")
cnn = Sequential([
    Embedding(input_dim=num_api+1, output_dim=32, input_length=100),
    Conv1D(64, 3, activation='relu'), MaxPooling1D(2),
    Conv1D(128, 3, activation='relu'), MaxPooling1D(2),
    Flatten(), Dense(64, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
cnn.fit(X_train, y_train, validation_data=(X_val, y_val),
        epochs=15, batch_size=256, verbose=0,
        callbacks=[EarlyStopping(monitor='val_auc', patience=3,
                                  restore_best_weights=True, mode='max')])
cnn_prob = cnn.predict(X_test, verbose=0).flatten()
_, auc_cnn, rep_cnn, auc_cnn_bal, rep_cnn_bal = evaluate_both(
    cnn_prob, X_test, y_test, X_test_bal, y_test_bal, bal_idx, is_model=False)
results['CNN'] = {'prob': cnn_prob.tolist(),
    'auc_full': auc_cnn, 'report_full': rep_cnn,
    'auc_bal': auc_cnn_bal, 'report_bal': rep_cnn_bal}
print(f"    CNN — Full AUC: {auc_cnn:.4f} | Balanced AUC: {auc_cnn_bal:.4f}")

# Feedforward
print("  Training Feedforward...")
ff = build_ff(num_api)
ff.fit(X_train, y_train, validation_data=(X_val, y_val),
       epochs=20, batch_size=256, verbose=0,
       callbacks=[EarlyStopping(monitor='val_auc', patience=5,
                                 restore_best_weights=True, mode='max')])
ff_prob = ff.predict(X_test, verbose=0).flatten()
_, auc_ff, rep_ff, auc_ff_bal, rep_ff_bal = evaluate_both(
    ff_prob, X_test, y_test, X_test_bal, y_test_bal, bal_idx, is_model=False)
results['Feedforward (Emb)'] = {'prob': ff_prob.tolist(),
    'auc_full': auc_ff, 'report_full': rep_ff,
    'auc_bal': auc_ff_bal, 'report_bal': rep_ff_bal}
print(f"    FF  — Full AUC: {auc_ff:.4f} | Balanced AUC: {auc_ff_bal:.4f}")

def print_table(title, auc_key, rep_key):
    print(f"\n{'='*76}\n{title}\n{'='*76}")
    print(f"{'Model':<24} {'AUC':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'BenF1':>8}")
    print("-"*76)
    for name, res in results.items():
        r = res[rep_key]
        ben_f1 = r.get('Benign', {}).get('f1-score', 0)
        marker = " ◄" if 'TEMPEST' in name else ""
        print(f"{name:<24} {res[auc_key]:>8.4f} {r['accuracy']:>8.4f} "
              f"{r['weighted avg']['precision']:>8.4f} "
              f"{r['weighted avg']['recall']:>8.4f} "
              f"{r['weighted avg']['f1-score']:>8.4f} "
              f"{ben_f1:>8.4f}{marker}")
    print("="*76)

print_table("TABLE II — FULL TEST SET", 'auc_full', 'report_full')
print_table("TABLE III — BALANCED TEST SET", 'auc_bal', 'report_bal')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: ABLATION STUDY
# ══════════════════════════════════════════════════════════════════════════════
print("\n[6/13] Running ablation study (same splits, same seed)...")
ablation = {}
ablation['TEMPEST: Emb + Stacked LSTM'] = {
    'auc': auc_main, 'accuracy': rep_main['accuracy'],
    'f1': rep_main['weighted avg']['f1-score']
}

def run_ablation_model(m, name):
    m.fit(X_train, y_train, validation_data=(X_val, y_val),
          epochs=20, batch_size=256, verbose=0,
          callbacks=[EarlyStopping(monitor='val_auc', patience=5,
                                    restore_best_weights=True, mode='max'),
                     ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                       patience=3, verbose=0)])
    p = m.predict(X_test, verbose=0).flatten()
    r = classification_report(y_test, (p>0.5).astype(int), output_dict=True)
    auc = roc_auc_score(y_test, p)
    ablation[name] = {'auc': auc, 'accuracy': r['accuracy'],
                       'f1': r['weighted avg']['f1-score']}
    print(f"    {name}: AUC={auc:.4f}")

m = Sequential([
    Embedding(input_dim=num_api+1, output_dim=32, input_length=100),
    LSTM(128), Dropout(0.3),
    Dense(32, activation='relu'), Dense(1, activation='sigmoid')
])
m.compile(optimizer='adam', loss='binary_crossentropy',
          metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
run_ablation_model(m, 'Emb + Single LSTM')

m = Sequential([
    Embedding(input_dim=num_api+1, output_dim=32, input_length=100),
    SimpleRNN(128, return_sequences=True), Dropout(0.3),
    SimpleRNN(64), Dropout(0.3),
    Dense(32, activation='relu'), Dense(1, activation='sigmoid')
])
m.compile(optimizer='adam', loss='binary_crossentropy',
          metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
run_ablation_model(m, 'Emb + SimpleRNN')

m = Sequential([
    tf.keras.layers.Reshape((100, 1), input_shape=(100,)),
    LSTM(128, return_sequences=True), BatchNormalization(), Dropout(0.3),
    LSTM(64), BatchNormalization(), Dropout(0.3),
    Dense(32, activation='relu'), Dense(1, activation='sigmoid')
])
m.compile(optimizer='adam', loss='binary_crossentropy',
          metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
run_ablation_model(m, 'LSTM (no Embedding)')

ablation['Feedforward (Emb)'] = {
    'auc': auc_ff, 'accuracy': rep_ff['accuracy'],
    'f1': rep_ff['weighted avg']['f1-score']
}

base_auc = ablation['TEMPEST: Emb + Stacked LSTM']['auc']
print(f"\n{'Variant':<32} {'AUC':>8} {'Acc':>8} {'F1':>8} {'ΔAUC':>10}")
print("-"*68)
for name, res in ablation.items():
    delta = res['auc'] - base_auc
    ds = f"{delta:+.2%}" if name != 'TEMPEST: Emb + Stacked LSTM' else "—"
    mark = " ◄" if 'TEMPEST' in name else ""
    print(f"{name:<32} {res['auc']:>8.4f} {res['accuracy']:>8.4f} "
          f"{res['f1']:>8.4f} {ds:>10}{mark}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7: SEQUENCE ORDER ABLATION
# ══════════════════════════════════════════════════════════════════════════════
print("\n[7/13] Sequence order experiments...")
np.random.seed(42)
X_test_shuf = np.array([np.random.permutation(x) for x in X_test])

prob_shuf    = model.predict(X_test_shuf, verbose=0).flatten()
auc_shuf     = roc_auc_score(y_test, prob_shuf)
ff_prob_shuf = ff.predict(X_test_shuf, verbose=0).flatten()
auc_ff_shuf  = roc_auc_score(y_test, ff_prob_shuf)

print(f"  TEMPEST original:   {auc_main:.4f}")
print(f"  TEMPEST shuffled:   {auc_shuf:.4f}  Δ={auc_shuf-auc_main:+.4f}  ({(auc_shuf-auc_main)/auc_main*100:+.2f}%)")
print(f"  FF original:        {auc_ff:.4f}")
print(f"  FF shuffled:        {auc_ff_shuf:.4f}  Δ={auc_ff_shuf-auc_ff:+.4f}  ({(auc_ff_shuf-auc_ff)/auc_ff*100:+.2f}%)")

shuffle_results = {
    'tempest_original': float(auc_main),   'tempest_shuffled': float(auc_shuf),
    'tempest_drop':     float(auc_main - auc_shuf),
    'tempest_drop_pct': float((auc_main - auc_shuf)/auc_main*100),
    'ff_original':      float(auc_ff),     'ff_shuffled': float(auc_ff_shuf),
    'ff_drop':          float(auc_ff - auc_ff_shuf),
    'ff_drop_pct':      float((auc_ff - auc_ff_shuf)/auc_ff*100),
}

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8: STATISTICAL VALIDITY — 15 SEEDS + WILCOXON SIGNED-RANK TEST
# ══════════════════════════════════════════════════════════════════════════════
# WHY WILCOXON:
#   The paired t-test assumes normally distributed differences, which is
#   unreliable with only 5 samples. Wilcoxon signed-rank is non-parametric:
#   it makes no distributional assumptions and is the standard choice when
#   n < 30. We also increase seeds from 5 → 15 to give the test more power.
# ══════════════════════════════════════════════════════════════════════════════
print("\n[8/13] Statistical validity — 15 seeds, Wilcoxon signed-rank test...")

SEEDS = [42, 123, 456, 789, 1337, 2024, 314, 999, 7, 2718,
         100, 200, 300, 400, 500]

seed_aucs_lstm = []
seed_aucs_ff   = []

for i, seed in enumerate(SEEDS):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    # ── LSTM ──
    m_l = build_lstm(num_api)
    m_l.fit(X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=30, batch_size=256, verbose=0,
            callbacks=[
                EarlyStopping(monitor='val_auc', patience=5,
                              restore_best_weights=True, mode='max'),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=3, verbose=0)
            ])
    p_l = m_l.predict(X_test, verbose=0).flatten()
    auc_l = float(roc_auc_score(y_test, p_l))
    seed_aucs_lstm.append(auc_l)

    # ── Feedforward ──
    m_f = build_ff(num_api)
    m_f.fit(X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=20, batch_size=256, verbose=0,
            callbacks=[
                EarlyStopping(monitor='val_auc', patience=5,
                              restore_best_weights=True, mode='max')
            ])
    p_f = m_f.predict(X_test, verbose=0).flatten()
    auc_f = float(roc_auc_score(y_test, p_f))
    seed_aucs_ff.append(auc_f)

    print(f"  Seed {seed:5d} [{i+1:2d}/15]: LSTM={auc_l:.4f} | FF={auc_f:.4f}  "
          f"diff={auc_l-auc_f:+.4f}")

# ── Wilcoxon signed-rank test ─────────────────────────────────────────────────
#   alternative='two-sided': tests whether the median difference is zero.
#   zero_method='wilcox': standard Wilcoxon method (drops zero differences).
w_stat, p_val_wilcox = stats.wilcoxon(seed_aucs_lstm, seed_aucs_ff,
                                       alternative='two-sided',
                                       zero_method='wilcox')

# Also keep t-test for comparison / reporting completeness
t_stat, p_val_t = stats.ttest_rel(seed_aucs_lstm, seed_aucs_ff)

mean_lstm = float(np.mean(seed_aucs_lstm))
std_lstm  = float(np.std(seed_aucs_lstm))
mean_ff   = float(np.mean(seed_aucs_ff))
std_ff    = float(np.std(seed_aucs_ff))
median_diff = float(np.median(np.array(seed_aucs_lstm) - np.array(seed_aucs_ff)))

print(f"\n  ── Results across {len(SEEDS)} seeds ──────────────────────────────")
print(f"  LSTM: {mean_lstm:.4f} ± {std_lstm:.4f}")
print(f"  FF:   {mean_ff:.4f} ± {std_ff:.4f}")
print(f"  Median LSTM−FF difference: {median_diff:+.4f}")
print(f"\n  Wilcoxon signed-rank: W={w_stat:.1f}, p={p_val_wilcox:.4f}")
print(f"  Paired t-test:        t={t_stat:.3f},  p={p_val_t:.4f}")
sig_wilcox = p_val_wilcox < 0.05
print(f"  Wilcoxon result: {'✓ Significant (p<0.05)' if sig_wilcox else '✗ Not significant (p≥0.05)'}")
print(f"\n  Interpretation: TEMPEST's contribution is temporally grounded")
print(f"  explainability, not raw AUC superiority. The AUC gap is {'significant' if sig_wilcox else 'not statistically significant'}.")

stat_results = {
    'n_seeds':        len(SEEDS),
    'seeds':          SEEDS,
    'lstm_aucs':      seed_aucs_lstm,
    'lstm_mean':      mean_lstm,
    'lstm_std':       std_lstm,
    'ff_aucs':        seed_aucs_ff,
    'ff_mean':        mean_ff,
    'ff_std':         std_ff,
    'median_diff':    median_diff,
    'wilcoxon_W':     float(w_stat),
    'wilcoxon_p':     float(p_val_wilcox),
    'wilcoxon_sig':   bool(sig_wilcox),
    't_stat':         float(t_stat),
    't_p_val':        float(p_val_t),
}

# ══════════════════════════════════════════════════════════════════════════════
# STEP 9: TRUNCATION CURVE
# ══════════════════════════════════════════════════════════════════════════════
print("\n[9/13] Truncation experiments...")
checkpoints = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
trunc_aucs  = []
for cp in checkpoints:
    m = build_lstm(num_api, seq_len=cp)
    m.fit(X_train[:, :cp], y_train,
          validation_data=(X_val[:, :cp], y_val),
          epochs=20, batch_size=256, verbose=0,
          callbacks=[EarlyStopping(monitor='val_auc', patience=5,
                                    restore_best_weights=True, mode='max')])
    p   = m.predict(X_test[:, :cp], verbose=0).flatten()
    auc = float(roc_auc_score(y_test, p))
    trunc_aucs.append(auc)
    print(f"  t_0–t_{cp-1:2d}: AUC={auc:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 10: INTEGRATED GRADIENTS + API NAME MAPPING
# ══════════════════════════════════════════════════════════════════════════════
print("\n[10/13] Computing Integrated Gradients...")

emb_layer   = model.layers[0]
post_layers = model.layers[1:]

def forward_from_emb(emb_tensor):
    x = emb_tensor
    for layer in post_layers:
        x = layer(x)
    return x

def compute_ig(X_int, steps=30):
    all_ig = []
    for i in range(len(X_int)):
        x_in  = tf.constant(X_int[i:i+1], dtype=tf.int32)
        emb   = emb_layer(x_in)
        base  = tf.zeros_like(emb)
        grads = []
        for alpha in np.linspace(0, 1, steps):
            interp = tf.Variable(base + alpha * (emb - base))
            with tf.GradientTape() as tape:
                tape.watch(interp)
                out = forward_from_emb(interp)
            g = tape.gradient(out, interp)
            if g is not None:
                grads.append(g.numpy())
        if not grads:
            all_ig.append(np.zeros(100))
            continue
        avg_g = np.mean(grads, axis=0)
        ig    = ((emb.numpy() - base.numpy()) * avg_g).sum(axis=-1)[0]
        all_ig.append(ig)
        if (i+1) % 50 == 0:
            print(f"  {i+1}/200 done...")
    return np.array(all_ig)

X_explain = X_test[100:300].astype(np.int32)
y_explain = y_test[100:300]
ig_values = compute_ig(X_explain, steps=30)
mean_ig   = np.abs(ig_values).mean(axis=0)
top20_idx = np.argsort(mean_ig)[-20:][::-1]

print("\n  TOP 20 INFLUENTIAL POSITIONS WITH API NAMES:")
print(f"  {'Rank':<6} {'Position':<12} {'Top API Call':<35} {'Mean|IG|'}")
print("  " + "-"*70)
api_name_findings = []
mal_mask = (y_test == 1)
for rank, pos in enumerate(top20_idx):
    ids_at_pos = [int(X_test[i, pos]) for i in range(len(X_test)) if mal_mask[i]]
    if ids_at_pos:
        most_common_id = Counter(ids_at_pos).most_common(1)[0][0]
        api_name = vocab_map.get(most_common_id,
                                  vocab_map.get(str(most_common_id),
                                                f"api_id_{most_common_id}"))
    else:
        api_name = "unknown"
    print(f"  {rank+1:<6} t_{int(pos):<10} {api_name:<35} {float(mean_ig[pos]):.6f}")
    api_name_findings.append({
        'rank': int(rank+1), 'position': int(pos),
        'api_name': str(api_name),
        'api_id': int(most_common_id) if ids_at_pos else -1,
        'mean_ig': float(mean_ig[pos])
    })

np.save('/kaggle/working/results/ig_values.npy', ig_values)
np.save('/kaggle/working/results/y_explain.npy', y_explain)
save_json(api_name_findings, '/kaggle/working/results/api_attribution_findings.json')
print("✅ IG saved.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 11: RUNTIME ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("\n[11/13] Runtime analysis...")
n_samp = 1000
X_s    = X_test[:n_samp]
runtime = {}
for name, obj, is_nn in [
    ('TEMPEST (LSTM)', model, True),
    ('Feedforward',    ff,    True),
    ('CNN',            cnn,   True),
    ('Random Forest',  rf,    False)
]:
    t0 = time.time()
    if is_nn: obj.predict(X_s, verbose=0)
    else:     obj.predict(X_s)
    runtime[name] = round((time.time()-t0)/n_samp*1000, 3)
    print(f"  {name:<22}: {runtime[name]:.3f} ms/sample")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 12: GENERATE ALL FIGURES
# ══════════════════════════════════════════════════════════════════════════════
print("\n[12/13] Generating figures...")

# ── Fig 1: ROC curves ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7,6))
colors = {'TEMPEST (Emb+LSTM)':'blue','Random Forest':'green',
          'SVM (RBF)':'orange','CNN':'purple','Feedforward (Emb)':'red'}
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    ax.plot(fpr, tpr, color=colors[name], lw=1.8,
            label=f"{name} (AUC={res['auc_full']:.4f})")
ax.plot([0,1],[0,1],'k--', lw=0.8)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
axins = ax.inset_axes([0.35, 0.08, 0.55, 0.45])
for name, res in results.items():
    if name == 'SVM (RBF)': continue
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    axins.plot(fpr, tpr, color=colors[name], lw=1.5)
axins.set_xlim(0, 0.05); axins.set_ylim(0.90, 1.0)
axins.set_title('Zoom: low FPR', fontsize=8)
axins.grid(True, alpha=0.3)
ax.indicate_inset_zoom(axins, edgecolor='gray')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/roc_all_models.png', dpi=200)
plt.close(); print("  Fig 1 (ROC) ✅")

# ── Fig 2: Truncation curve ───────────────────────────────────────────────────
plt.figure(figsize=(9,5))
plt.plot(checkpoints, trunc_aucs, 'b-o', lw=2, ms=7, label='AUC by length', zorder=3)
for cp, auc in zip(checkpoints, trunc_aucs):
    plt.annotate(f'{auc:.3f}', (cp, auc), textcoords="offset points",
                 xytext=(0,8), ha='center', fontsize=7.5)
plt.axhline(y=auc_main, color='gray', ls='--', lw=1.5,
            label=f'Full sequence ({auc_main:.4f})')
plt.axhline(y=float(auc_shuf), color='red', ls='--', lw=1.5,
            label=f'Shuffled ({auc_shuf:.4f})')
plt.xlabel('API Call Positions Visible ($t_0$ to $t_n$)', fontsize=12)
plt.ylabel('AUC-ROC', fontsize=12)
plt.title('TEMPEST — Detection Performance vs. Sequence Length', fontsize=12)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/sequence_importance.png', dpi=200)
plt.close(); print("  Fig 2 (Truncation) ✅")

# ── Fig 3: IG full sequence ───────────────────────────────────────────────────
plt.figure(figsize=(12,4))
plt.plot(mean_ig, color='crimson', lw=1.8)
plt.fill_between(range(100), mean_ig, alpha=0.25, color='crimson')
plt.axvspan(75, 99, alpha=0.1, color='red', label='Late phase (t75–t99)')
plt.xlabel('API Call Position ($t_0$ to $t_{99}$)', fontsize=12)
plt.ylabel('Mean |Integrated Gradient|', fontsize=12)
plt.title('TEMPEST — Feature Importance Across Full API Call Sequence', fontsize=12)
plt.legend(fontsize=9); plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/ig_sequence.png', dpi=200)
plt.close(); print("  Fig 3 (IG full) ✅")

# ── Fig 4: IG malware vs benign ───────────────────────────────────────────────
mal_idx = np.where(y_explain == 1)[0]
ben_idx = np.where(y_explain == 0)[0]
mal_ig  = np.abs(ig_values[mal_idx]).mean(axis=0) if len(mal_idx) else np.zeros(100)
ben_ig  = np.abs(ig_values[ben_idx]).mean(axis=0) if len(ben_idx) else np.zeros(100)
plt.figure(figsize=(12,4))
plt.plot(mal_ig, color='red',   lw=1.8, label=f'Malware (n={len(mal_idx)})')
plt.plot(ben_ig, color='green', lw=1.8, label=f'Benign  (n={len(ben_idx)})')
plt.xlabel('API Call Position', fontsize=12)
plt.ylabel('Mean |Integrated Gradient|', fontsize=12)
plt.title('TEMPEST — Feature Importance: Malware vs Benign', fontsize=12)
plt.legend(fontsize=10); plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/ig_mal_vs_ben.png', dpi=200)
plt.close(); print("  Fig 4 (IG mal vs ben) ✅")

# ── Fig 5: IG single sample ───────────────────────────────────────────────────
s = int(mal_idx[0]) if len(mal_idx) > 0 else 0
plt.figure(figsize=(12,4))
bar_colors = ['red' if v > 0 else 'steelblue' for v in ig_values[s]]
plt.bar(range(100), ig_values[s], color=bar_colors, width=0.8)
plt.axhline(0, color='black', lw=0.5)
plt.xlabel('API Call Position ($t_0$ to $t_{99}$)', fontsize=12)
plt.ylabel('Integrated Gradient', fontsize=12)
plt.title('TEMPEST — Single Malware Sample: Which Positions Triggered the Alert', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/ig_single_sample.png', dpi=200)
plt.close(); print("  Fig 5 (IG single) ✅")

# ── Fig 6: Top 20 positions ───────────────────────────────────────────────────
plt.figure(figsize=(13,5))
top20_vals   = mean_ig[top20_idx]
top20_labels = [f"t_{p}\n({api_name_findings[i]['api_name'][:12]})"
                for i, p in enumerate(top20_idx)]
plt.bar(range(20), top20_vals, color='steelblue', edgecolor='white')
plt.xticks(range(20), top20_labels, rotation=45, ha='right', fontsize=8)
plt.xlabel('API Call Position (with most common API name)', fontsize=11)
plt.ylabel('Mean |Integrated Gradient|', fontsize=11)
plt.title('TEMPEST — Top 20 Most Influential API Call Positions', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/ig_top20.png', dpi=200)
plt.close(); print("  Fig 6 (Top 20) ✅")

# ── Fig 7a/7b: Confusion matrices ─────────────────────────────────────────────
for tag, y_t, probs, fname in [
    ('Full',     y_test,     prob_main,              'confusion_matrix_full.png'),
    ('Balanced', y_test_bal, prob_main[bal_idx],     'confusion_matrix_balanced.png')
]:
    cm = confusion_matrix(y_t, (probs>0.5).astype(int))
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign','Malware'],
                yticklabels=['Benign','Malware'], annot_kws={'size':13})
    plt.title(f'TEMPEST — Confusion Matrix ({tag})', fontsize=12)
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/figures/{fname}', dpi=200)
    plt.close()
print("  Fig 7a/7b (Confusion matrices) ✅")

# ── Fig 8: Training history ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(history.history['loss'],     label='Train', color='blue')
ax1.plot(history.history['val_loss'], label='Val',   color='orange')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history.history['auc'],     label='Train', color='blue')
ax2.plot(history.history['val_auc'], label='Val',   color='orange')
ax2.set_title('AUC'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle('TEMPEST Training History', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/training_history.png', dpi=200)
plt.close(); print("  Fig 8 (Training history) ✅")

# ── Fig 9: Statistical comparison — 15 seeds, Wilcoxon ───────────────────────
fig, ax = plt.subplots(figsize=(7,5))
bp = ax.boxplot([seed_aucs_lstm, seed_aucs_ff],
                labels=['TEMPEST\n(LSTM)', 'Feedforward\n(Emb)'],
                patch_artist=True, widths=0.4)
bp['boxes'][0].set(facecolor='steelblue', alpha=0.7)
bp['boxes'][1].set(facecolor='coral',     alpha=0.7)
ax.scatter([1]*len(seed_aucs_lstm), seed_aucs_lstm,
           color='navy',  zorder=3, s=45, label='Individual seeds')
ax.scatter([2]*len(seed_aucs_ff),   seed_aucs_ff,
           color='firebrick', zorder=3, s=45)
sig_label = (f"Wilcoxon: W={w_stat:.0f}, p={p_val_wilcox:.3f} "
             f"({'sig.' if sig_wilcox else 'n.s.'})")
ax.set_title(f'Statistical Comparison ({len(SEEDS)} seeds)\n{sig_label}', fontsize=11)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
# Annotate mean ± std
for xi, (vals, c) in enumerate([(seed_aucs_lstm,'navy'),(seed_aucs_ff,'firebrick')], 1):
    ax.text(xi, min(vals)-0.0015,
            f"μ={np.mean(vals):.4f}\nσ={np.std(vals):.4f}",
            ha='center', va='top', fontsize=8, color=c)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/statistical_comparison.png', dpi=200)
plt.close(); print("  Fig 9 (Statistical comparison) ✅")

# ── Fig 10: Full vs Balanced accuracy bar ────────────────────────────────────
model_names = list(results.keys())
accs_full   = [results[m]['report_full']['accuracy'] for m in model_names]
accs_bal    = [results[m]['report_bal']['accuracy']  for m in model_names]
x = np.arange(len(model_names)); w = 0.35
fig, ax = plt.subplots(figsize=(11,5))
b1 = ax.bar(x-w/2, accs_full, w, label='Full test set',     color='steelblue')
b2 = ax.bar(x+w/2, accs_bal,  w, label='Balanced test set', color='coral')
ax.set_ylim(0.5, 1.05)
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylabel('Accuracy'); ax.legend()
ax.set_title('Accuracy: Full vs Balanced Test Set\n'
             '(Balanced removes inflated majority-class accuracy)')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=7)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=7)
ax.axhline(0.5, color='red', ls=':', lw=1, label='"Flag all" baseline on balanced')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/full_vs_balanced_accuracy.png', dpi=200)
plt.close(); print("  Fig 10 (Full vs Balanced) ✅")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 13: SAVE ALL RESULTS TO JSON
# ══════════════════════════════════════════════════════════════════════════════
print("\n[13/13] Saving all results...")

all_results = {
    'dataset': {
        'total_samples':      int(len(X)),
        'malware':            int(y.sum()),
        'benign':             int((y==0).sum()),
        'test_malware':       int(y_test.sum()),
        'test_benign':        int((y_test==0).sum()),
        'balanced_test_each': int(n_ben),
        'flagall_full_acc':   float(y_test.sum()/len(y_test)),
        'flagall_bal_acc':    0.5,
    },
    'main_model': {
        'auc_full':           float(auc_main),
        'auc_balanced':       float(auc_main_bal),
        'accuracy_full':      float(rep_main['accuracy']),
        'accuracy_balanced':  float(rep_main_bal['accuracy']),
        'f1_full':            float(rep_main['weighted avg']['f1-score']),
        'benign_f1_full':     float(rep_main.get('Benign',{}).get('f1-score',0)),
        'benign_f1_balanced': float(rep_main_bal.get('Benign',{}).get('f1-score',0)),
        'malware_f1_full':    float(rep_main.get('Malware',{}).get('f1-score',0)),
    },
    'baselines_full': {
        k: {'auc':      float(v['auc_full']),
            'accuracy': float(v['report_full']['accuracy']),
            'f1':       float(v['report_full']['weighted avg']['f1-score'])}
        for k,v in results.items()
    },
    'baselines_balanced': {
        k: {'auc':      float(v['auc_bal']),
            'accuracy': float(v['report_bal']['accuracy']),
            'f1':       float(v['report_bal']['weighted avg']['f1-score']),
            'benign_f1':float(v['report_bal'].get('Benign',{}).get('f1-score',0))}
        for k,v in results.items()
    },
    'ablation': {
        k: {kk: float(vv) for kk,vv in v.items()}
        for k,v in ablation.items()
    },
    'sequence_order': shuffle_results,
    'statistical':    stat_results,
    'truncation':     {f't0-t{c-1}': float(a) for c,a in zip(checkpoints, trunc_aucs)},
    'runtime_ms':     {k: float(v) for k,v in runtime.items()},
    'api_attribution_top20': api_name_findings,
}

save_json(all_results, '/kaggle/working/results/all_results.json')

# ── FINAL SUMMARY PRINT ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(f"\nMain model — Full test:     AUC={auc_main:.4f} | Acc={rep_main['accuracy']:.4f}")
print(f"Main model — Balanced test: AUC={auc_main_bal:.4f} | Acc={rep_main_bal['accuracy']:.4f}")
print(f"  Benign F1 (balanced):     {rep_main_bal['Benign']['f1-score']:.4f}")
print(f"  'Flag all' baseline:      {y_test.sum()/len(y_test):.1%} (full) | 50.0% (balanced)")

print(f"\nSequence order ablation:")
print(f"  TEMPEST drop: {auc_main:.4f} → {auc_shuf:.4f}  ({(auc_shuf-auc_main)/auc_main*100:+.2f}%)")
print(f"  FF drop:      {auc_ff:.4f} → {auc_ff_shuf:.4f}  ({(auc_ff_shuf-auc_ff)/auc_ff*100:+.2f}%)")

print(f"\nStatistical ({len(SEEDS)} seeds):")
print(f"  LSTM: {mean_lstm:.4f} ± {std_lstm:.4f}")
print(f"  FF:   {mean_ff:.4f} ± {std_ff:.4f}")
print(f"  Wilcoxon: W={w_stat:.1f}, p={p_val_wilcox:.4f} — "
      f"{'significant' if sig_wilcox else 'not significant (emphasise explainability)'}")

print(f"\nFigures saved:  {len(os.listdir('/kaggle/working/figures'))} files")
print(f"Results JSON:   /kaggle/working/results/all_results.json")

print("\n" + "="*70)
print("🎉 COMPLETE — Ready to update paper with new numbers")
print("="*70)

2026-04-10 10:49:57.736033: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775818197.944888      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775818198.007031      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775818198.495035      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775818198.495077      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775818198.495080      24 computation_placer.cc:177] computation placer alr

TEMPEST MASTER CELL v2 — Full pipeline

[1/13] Downloading datasets...
Dataset URL: https://www.kaggle.com/datasets/ang3loliveira/malware-analysis-datasets-api-call-sequences
License(s): Attribution 4.0 International (CC BY 4.0)
Dataset URL: https://www.kaggle.com/datasets/gautamkarat/api-call-sequences
License(s): CC0-1.0
✅ Done.

[2/13] Building API vocabulary...
  Could not build vocabulary: [Errno 2] No such file or directory: '/kaggle/working/data/raw/sample_analysis_data.txt'

[3/13] Loading and preprocessing...
  Combined: (47816, 100) | Malware: 45423 | Benign: 2393
  Vocab size: 615
  Test — Malware: 9085 | Benign: 479
  Applying SMOTE on training data only...
  Train after SMOTE: (65406, 100)
  Balanced test — 479 malware + 479 benign = 958 total
  'Flag everything' baseline on balanced test: 50.0%
✅ Splits saved.

[4/13] Training main TEMPEST model...


I0000 00:00:1775818228.259827      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775818228.266278      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

Epoch 1/30


I0000 00:00:1775818234.403526      77 cuda_dnn.cc:529] Loaded cuDNN version 91002


256/256 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.9173 - auc: 0.9634 - loss: 0.2159 - precision: 0.9186 - recall: 0.9195 - val_accuracy: 0.8743 - val_auc: 0.9047 - val_loss: 0.4756 - val_precision: 0.9901 - val_recall: 0.8765 - learning_rate: 0.0010
Epoch 2/30
256/256 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9764 - auc: 0.9943 - loss: 0.0774 - precision: 0.9756 - recall: 0.9775 - val_accuracy: 0.9757 - val_auc: 0.9363 - val_loss: 0.1093 - val_precision: 0.9852 - val_recall: 0.9893 - learning_rate: 0.0010
Epoch 3/30
256/256 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9833 - auc: 0.9959 - loss: 0.0585 - precision: 0.9815 - recall: 0.9854 - val_accuracy: 0.9754 - val_auc: 0.9528 - val_loss: 0.0902 - val_precision: 0.9865 - val_recall: 0.9876 - learning_rate: 0.0010
Epoch 4/30
256/256 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.9874 - auc: 0.9967 - loss: 0.0472 - precision: 0.9864 - recall: 0.9885 - val_accuracy: 0.9749 - val_auc: 0.9739 - val_loss: 0.0861 - val_preci

I0000 00:00:1775818355.719462      77 service.cc:152] XLA service 0x7d9b28e74a10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775818355.719508      77 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775818355.719512      77 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775818360.139397      77 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    CNN — Full AUC: 0.9907 | Balanced AUC: 0.9915
  Training Feedforward...
    FF  — Full AUC: 0.9916 | Balanced AUC: 0.9919

TABLE II — FULL TEST SET
Model                         AUC      Acc     Prec      Rec       F1    BenF1
----------------------------------------------------------------------------
TEMPEST (Emb+LSTM)         0.9846   0.9825   0.9823   0.9825   0.9824   0.8233 ◄
Random Forest              0.9904   0.9819   0.9812   0.9819   0.9814   0.8097
SVM (RBF)                  0.9411   0.9240   0.9607   0.9240   0.9370   0.5264
CNN                        0.9907   0.9868   0.9865   0.9868   0.9866   0.8645
Feedforward (Emb)          0.9916   0.9860   0.9858   0.9860   0.9859   0.8581

TABLE III — BALANCED TEST SET
Model                         AUC      Acc     Prec      Rec       F1    BenF1
----------------------------------------------------------------------------
TEMPEST (Emb+LSTM)         0.9852   0.9019   0.9153   0.9019   0.9011   0.8922 ◄
Random Forest              